In [ ]:
import torch
print("CUDA Available: ", torch.cuda.is_available())
print("CUDA Device Name: ", torch.cuda.get_device_name(0))
torch.cuda.empty_cache()

# Verify CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

In [ ]:
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from app.utils.settings import COLLECTION_NAME, CHUNKS_FILE
from app.utils.chunking import load_chunks
from app.ingest.qdrant_factory import QdrantFactory

In [ ]:
def ingest_chunks_to_qdrant():
    """
    Ingests pre-chunked documents into Qdrant vector store using local embeddings.
    Creates the collection if it doesn't exist and adds texts + metadata in batch.
    """
    # 1. Initialize factory with device
    factory = QdrantFactory(device=device)

    # 2. Use factory to get client and vector store
    client = factory.client
    vector_store = factory.get_qdrant_vector_store()

    # 3. Create collection if it doesn't exist (fixed dimension + named vectors)
    if not client.collection_exists(collection_name=COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "text-dense": VectorParams(size=384, distance=Distance.COSINE)
            },
            sparse_vectors_config={
                "text-sparse": SparseVectorParams()  # no size needed for sparse vectors
            },
        )
        print(f"Collection '{COLLECTION_NAME}' created with 'text-dense' vector.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")

    # 4. Load chunks from pickle file (saved by extract_3gpp.ipynb)
    chunks = load_chunks(CHUNKS_FILE)
    if not chunks:
        print("No chunks found to ingest.")
        return

    # 5. Prepare texts and metadatas
    texts = []
    metadatas = []

    for index, chunk in enumerate(chunks):
        text = chunk.get("content", "")
        if not text:
            continue

        texts.append(text)

        metadata = {
            "release": chunk.get("release", ""),
            "series": chunk.get("series", ""),
            "spec": chunk.get("spec", ""),
            "chunk_index": index,  # opcional
        }
        metadatas.append(metadata)

    if not texts:
        print("No valid texts found after processing.")
        return

    # 6. Batch ingest
    vector_store.add_texts(texts=texts, metadatas=metadatas)
    print(f"Ingested {len(texts)} chunks into collection '{COLLECTION_NAME}'.")

In [ ]:
# Run ingestion
ingest_chunks_to_qdrant()

# Test collection

In [ ]:
factory = QdrantFactory(device=device)
client = factory.client
collection_info = client.get_collection(COLLECTION_NAME)
print(collection_info)
print(f"\nCollection: {COLLECTION_NAME}")
print(f"Status: {collection_info.status}")
print(f"Points count: {collection_info.points_count:,}")